In [3]:
# -----------------------------
# Section 1: Search Mechanisms
# -----------------------------

# 1. Problem Representation
# - A search problem can be represented as a graph or state space
# - Nodes = states, Edges = possible actions
# - Goal: find path from start to goal
# - Python adjacency list representation
graph = {
    'S': ['A', 'D'],
    'A': ['B', 'C'],
    'B': ['C', 'G'],
    'C': ['G'],
    'D': ['B', 'E'],
    'E': ['G'],
    'G': []  # goal node
}

# -----------------------------
# 2. Depth-First Search (DFS)
# -----------------------------
# - Explores one branch deeply before backtracking
# - Uses recursion or stack
# - Memory efficient but may not find shortest path
# - Must track visited nodes to avoid cycles

def dfs(graph, start, goal, path=None, visited=None):
    if path is None:
        path = [start]          # current path
    if visited is None:
        visited = set([start])  # visited nodes
    
    if start == goal:
        return path            # base case: reached goal
    
    for neighbor in graph.get(start, []):
        if neighbor not in visited:
            visited.add(neighbor)
            result = dfs(graph, neighbor, goal, path + [neighbor], visited)
            if result is not None:
                return result
    return None  # no path found

# Example run
path = dfs(graph, 'S', 'G')
print("DFS path:", path)
# Output depends on neighbor order, e.g., ['S', 'A', 'B', 'C', 'G']

DFS path: ['S', 'A', 'B', 'C', 'G']


In [4]:
# -----------------------------
# 3. DFS to Find All Paths
# -----------------------------
# - Instead of returning first path, accumulate all paths
def dfs_all_paths(graph, start, goal, path=None, visited=None):
    if path is None:
        path = [start]
    if visited is None:
        visited = set([start])
    
    if start == goal:
        return [path]  # return list of paths
    
    all_paths = []
    for neighbor in graph.get(start, []):
        if neighbor not in visited:
            visited.add(neighbor)
            new_paths = dfs_all_paths(graph, neighbor, goal, path + [neighbor], visited)
            all_paths.extend(new_paths)
            visited.remove(neighbor)  # allow node in other paths
    return all_paths

all_paths = dfs_all_paths(graph, 'S', 'G')
print("All DFS paths from S to G:")
for p in all_paths:
    print(p)


All DFS paths from S to G:
['S', 'A', 'B', 'C', 'G']
['S', 'A', 'B', 'G']
['S', 'A', 'C', 'G']
['S', 'D', 'B', 'C', 'G']
['S', 'D', 'B', 'G']
['S', 'D', 'E', 'G']


In [4]:
#4. DFS for Tower of Hanoi (3 disks)
# -----------------------------
# - State: list of 3 pegs with disks (largest at bottom)
# - Goal: move all disks from peg1 to peg3
# - Legal move: top disk to empty peg or larger top disk

def hanoi_dfs(state, goal, path=None, visited=None):
    if path is None:
        path = [state]
    if visited is None:
        visited = set()
    
    state_key = tuple(tuple(peg) for peg in state)  # make hashable
    if state_key in visited:
        return None
    visited.add(state_key)
    
    if state == goal:
        return path
    
    for i in range(3):
        for j in range(3):
            if i != j and state[i]:  # source peg not empty
                disk = state[i][-1]
                if not state[j] or state[j][-1] > disk:  # legal move
                    new_state = [list(peg) for peg in state]
                    new_state[i].pop()
                    new_state[j].append(disk)
                    result = hanoi_dfs(new_state, goal, path + [new_state], visited)
                    if result:
                        return result
    return None

# Example run
start = [[3,2,1], [], []]
goal = [[], [], [3,2,1]]
solution = hanoi_dfs(start, goal)
print("Tower of Hanoi solution steps:")
for step in solution:
    print(step)

Tower of Hanoi solution steps:
[[3, 2, 1], [], []]
[[3, 2], [1], []]
[[3], [1], [2]]
[[3, 1], [], [2]]
[[3], [], [2, 1]]
[[], [3], [2, 1]]
[[1], [3], [2]]
[[], [3, 1], [2]]
[[2], [3, 1], []]
[[2, 1], [3], []]
[[2, 1], [], [3]]
[[2], [1], [3]]
[[], [1], [3, 2]]
[[1], [], [3, 2]]
[[], [], [3, 2, 1]]


In [6]:
# -----------------------------
# Section 2: Breadth-First Search (BFS)
# -----------------------------

# BFS explores the search tree level by level
# - Uses a FIFO queue
# - Complete: guarantees shortest path if all edges have equal cost
# - High memory usage because all nodes at current level are stored

from collections import deque
import heapq

# -----------------------------
# 1. BFS Implementation
# -----------------------------
def bfs(graph, start, goal):
    queue = deque([(start, [start])])  # store tuples: (current_node, path)
    visited = set([start])             # track visited nodes

    while queue:
        current, path = queue.popleft()   # FIFO
        if current == goal:
            return path                  # goal reached
        for neighbor in graph.get(current, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return None

# Example graph
graph = {
    'S': ['A', 'D'],
    'A': ['B', 'C'],
    'B': ['C', 'G'],
    'C': ['G'],
    'D': ['B', 'E'],
    'E': ['G'],
    'G': []
}

path_bfs = bfs(graph, 'S', 'G')
print("BFS path:", path_bfs) 

BFS path: ['S', 'A', 'B', 'G']


In [ ]:
# -----------------------------
# 2. Performance Comparison Notes
# -----------------------------
# DFS vs BFS:
# - DFS: low memory, may find longer path, uses stack/recursion
# - BFS: higher memory, guarantees shortest path, uses queue
# Trade-off: BFS better for shortest path, DFS better for deep search or limited memory

In [8]:
# -----------------------------
# 3. Uniform Cost Search (UCS)
# -----------------------------
# UCS expands nodes based on lowest cumulative cost using a priority queue
def ucs(graph, start, goal):
    # graph should have edges with cost: e.g., {'S': [('A',1),('D',2)]}
    queue = [(0, start, [start])]  # (cumulative_cost, node, path)
    visited = set()

    while queue:
        cost, current, path = heapq.heappop(queue)
        if current == goal:
            return path, cost
        if current in visited:
            continue
        visited.add(current)

        for neighbor, edge_cost in graph.get(current, []):
            if neighbor not in visited:
                heapq.heappush(queue, (cost + edge_cost, neighbor, path + [neighbor]))
    return None, None

# Example weighted graph
graph_weighted = {
    'S': [('A', 1), ('D', 2)],
    'A': [('B', 2), ('C', 2)],
    'B': [('C', 1), ('G', 5)],
    'C': [('G', 1)],
    'D': [('B', 2), ('E', 3)],
    'E': [('G', 1)],
    'G': []
}

path_ucs, total_cost = ucs(graph_weighted, 'S', 'G')
print("UCS path:", path_ucs, "with total cost:", total_cost)

UCS path: ['S', 'A', 'C', 'G'] with total cost: 4


In [ ]:
# -----------------------------
# 4. Discussion Questions (Short Notes)
# -----------------------------
# 1. When DFS is preferred over BFS:
#    - Memory limited, deep/infinite search space, any solution is enough
# 2. Why BFS guarantees shortest path, DFS does not:
#    - BFS uses queue (level-order) → first reach goal = fewest edges
#    - DFS uses stack/recursion → may go deep before goal → longer path
# 3. Real-world applications:
#    - BFS/UCS for shortest paths (maps, routing)
#    - A* search with heuristics improves performance (fewer nodes expanded)

In [26]:
def forward_chaining(rules, facts):
    """
    Perform forward chaining on a set of rules and initial facts.
    :param rules: list of dicts with 'conditions' (set) and 'action' (string)
    :param facts: set of initial facts
    :return: expanded set of facts
    """
    inferred = True
    while inferred:
        inferred = False
        for rule in rules:
            # apply rule if all conditions are met and the action is new
            if rule['conditions'].issubset(facts) and rule['action'] not in facts:
                facts.add(rule['action'])
                inferred = True
    # return after all rules are applied
    return facts

# Example knowledge base
rules = [
    {'conditions': {'fever', 'cough'}, 'action': 'flu'},
    {'conditions': {'sore_throat'}, 'action': 'throat_infection'},
    {'conditions': {'flu'}, 'action': 'give_paracetamol'}
]

initial_facts = {'sore_throat'}
all_facts = forward_chaining(rules, set(initial_facts))
print("Derived facts:", all_facts)


Derived facts: {'sore_throat', 'throat_infection'}


In [45]:
def forward_chaining(rules, facts):
    """
    Perform forward chaining on a set of rules and initial facts.
    :param rules: list of dicts with 'conditions' (set) and 'action' (string)
    :param facts: set of initial facts
    :return: expanded set of facts including inferred ones
    """
    inferred = True
    while inferred:
        inferred = False
        for rule in rules:
            # Apply rule if all conditions are met and the action is new
            if rule['conditions'].issubset(facts) and rule['action'] not in facts:
                facts.add(rule['action'])
                inferred = True
    return facts

# Extended knowledge base
rules = [
    {'conditions': {'fever', 'cough'}, 'action': 'flu'},
    {'conditions': {'sore_throat'}, 'action': 'throat_infection'},
    {'conditions': {'flu'}, 'action': 'give_paracetamol'},
    {'conditions': {'flu', 'sore_throat'}, 'action': 'throat_culture'},
    {'conditions': {'throat_infection'}, 'action': 'give_antibiotics'}
]

# Test with new facts
initial_facts = {'flu','cough'}
all_facts = forward_chaining(rules, set(initial_facts))
print("Derived facts:", all_facts)


Derived facts: {'flu', 'cough', 'give_paracetamol'}


In [40]:
def forward_chaining_safe(rules, facts):
    """
    Perform forward chaining with cycle prevention.
    Each rule fires at most once, so cycles cannot cause an infinite loop.

    :param rules: list of dicts with 'conditions' (set) and 'action' (string)
    :param facts: set of initial facts
    :return: expanded set of facts including inferred ones
    """
    fired_rules = set()  # Keep track of rules that have already fired
    inferred = True

    while inferred:
        inferred = False
        for i, rule in enumerate(rules):
            # Only fire a rule if it has not fired before
            if i not in fired_rules and rule['conditions'].issubset(facts):
                if rule['action'] not in facts:
                    facts.add(rule['action'])
                fired_rules.add(i)
                inferred = True

    return facts

# Example rules (including a cycle)
rules = [
    {'conditions': {'A'}, 'action': 'B'},
    {'conditions': {'B'}, 'action': 'A'},
    {'conditions': {'fever', 'cough'}, 'action': 'flu'},
    {'conditions': {'sore_throat'}, 'action': 'throat_infection'}
]

# Initial facts
initial_facts = {'A', 'fever', 'cough', 'sore_throat'}

# Run forward chaining safely
all_facts = forward_chaining_safe(rules, set(initial_facts))
print("Derived facts:", all_facts)


Derived facts: {'cough', 'B', 'sore_throat', 'flu', 'A', 'throat_infection', 'fever'}
